# WP26 — Hierarchical Task Decomposition
## Layer 9: Decomposing Puzzle Batches into Subtask Types

---

This notebook demonstrates **WP26: Hierarchical Task Decomposition**, the ninth layer
of the Prometheus synthesis stack. Rather than treating every generation as a flat
sequence of indistinguishable puzzles, WP26 classifies puzzles into structural types
and applies specialised synthesis recommendations per subtask.

### Layer Hierarchy After WP26

| Layer | Work Package | Mechanism |
|-------|-------------|----------|
| 9 | **WP26 TaskDecomposer** | Per-subtask synthesis recommendation |
| 8 | WP25 EWC Guard | Catastrophic forgetting prevention |
| 7 | WP24 Ensemble JSD | Epistemic uncertainty gate |
| 6 | WP23 Student Policy | Knowledge distillation |
| 5 | WP22 Bandit | Explore/exploit |
| 4 | WP21 Meta-Gradient | Hyperparameter self-optimisation |
| 3 | WP20 Rollout Planner | K-step trajectory lookahead |
| 2 | WP19 Causal ATE | Potential-outcomes action evaluation |
| 1 | WP17 Heuristic | Demote/Promote/Reset/Boost |

### Theoretical Grounding
> *"A planner that cannot decompose a goal into subgoals will repeatedly
> re-examine identical subproblems."* — E.D. Sacerdoti (1977)

Runtime: **~5 min** (no GPU required)

In [ ]:
# ── 0. Environment setup ────────────────────────────────────────────────────
import sys, os, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
    print(f'Local mode — repo root: {repo_root}')

import warnings; warnings.filterwarnings('ignore')
import time, random, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

from prometheus.wp26_hierarchical_decomp import (
    HierarchicalCRLS, TaskDecomposer, SubtaskType,
    SubtaskRecord, DecompositionRecord,
    verify_wp26_exit_criteria,
)
from prometheus.wp22_bandit_exploration import BanditMode
from prometheus.wp17_crls_synthesis import SynthesisAction
from prometheus.environments.go import GoBoard

import prometheus
print(f'Prometheus version: {prometheus.__version__}')
print('WP26 imports OK.')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)

In [ ]:
# ── 1. Configuration ────────────────────────────────────────────────────────
QUICK_MODE       = True
N_GENERATIONS    = 10 if QUICK_MODE else 30
PUZZLES_PER_GEN  = 30 if QUICK_MODE else 80
BOARD_SIZE       = 9
BANDIT_MODE      = BanditMode.UCB1

# Regime sequence — mixed types exercise the decomposer
REGIME_SEQUENCE = (
    ['ATARI']    * 3 +
    ['TERRITORY']* 3 +
    ['MIXED']    * (N_GENERATIONS - 6)
)[:N_GENERATIONS]

print(f'Mode: {"QUICK" if QUICK_MODE else "FULL"}')
print(f'Generations: {N_GENERATIONS} | Puzzles/gen: {PUZZLES_PER_GEN}')
print(f'Regimes: {REGIME_SEQUENCE}')

---
## Section 1 — Puzzle Engine & Subtask Type Detection

`TaskDecomposer` classifies each puzzle into one of four `SubtaskType` values:
- **CAPTURE** — win by capturing opponent stones (ATARI / LADDER puzzles)
- **TERRITORY** — win by spatial anchoring
- **MIXED** — multiple objectives
- **UNKNOWN** — cannot be classified

In [ ]:
# ── 2. Puzzle factory (reuse from WP25 demo pattern) ────────────────────────
def make_atari_puzzle(board_size, rng):
    board = GoBoard(size=board_size)
    cx = board_size // 2
    stones = [(cx, cx), (cx, cx+1), (cx+1, cx)]
    for r, c in stones:
        if board.is_on_board(r, c) and board.board[r, c] == GoBoard.EMPTY:
            board.board[r, c] = GoBoard.BLACK
    all_libs = set()
    for r, c in stones:
        for nr, nc in board.get_neighbors(r, c):
            if board.board[nr, nc] == GoBoard.EMPTY:
                all_libs.add((nr, nc))
    libs = list(all_libs); rng.shuffle(libs)
    for r, c in libs[:-1]:
        board.board[r, c] = GoBoard.WHITE
    target = libs[-1]
    board.current_player = GoBoard.WHITE
    return board, GoBoard.WHITE, target

def make_territory_puzzle(board_size, rng):
    board = GoBoard(size=board_size)
    for r, c in [(0, 0), (0, board_size-1), (board_size-1, 0)]:
        board.board[r, c] = GoBoard.WHITE
    target = (board_size-1, board_size-1)
    board.current_player = GoBoard.BLACK
    return board, GoBoard.BLACK, target

PUZZLE_FACTORIES = {'ATARI': make_atari_puzzle, 'TERRITORY': make_territory_puzzle}

def generate_puzzles(regime, n, board_size, seed):
    rng = np.random.default_rng(seed)
    if regime == 'MIXED':
        return [
            (PUZZLE_FACTORIES['ATARI'] if i % 2 == 0 else PUZZLE_FACTORIES['TERRITORY'])(board_size, rng)
            for i in range(n)
        ]
    return [PUZZLE_FACTORIES[regime](board_size, rng) for _ in range(n)]

def evaluate_move(board, move, correct_move, player):
    if move == correct_move: return True
    if board.is_legal_move(move[0], move[1], player):
        return len(board.would_capture(move[0], move[1], player)) > 0
    return False

rng_test = np.random.default_rng(0)
b, p, m = make_atari_puzzle(BOARD_SIZE, rng_test)
print(f'Puzzle factory OK. Atari target: {m}, legal: {b.is_legal_move(m[0], m[1], p)}')

---
## Section 2 — Instantiate HierarchicalCRLS (9 Layers)

`HierarchicalCRLS` extends `EWCEnsembleCRLS` (WP25) with a `TaskDecomposer` at layer 9.
The 9-tuple returned by `end_of_generation()` includes a `DecompositionRecord` as the
final element.

In [ ]:
# ── 3. Instantiate 9-layer HierarchicalCRLS ──────────────────────────────────
STRATEGIES = ['ATARI', 'LADDER', 'TERRITORY', 'MIXED']

stack = HierarchicalCRLS(
    strategies                = STRATEGIES,
    bandit_mode               = BANDIT_MODE,
    decomp_confidence_base    = 0.80,
    decomp_override_threshold = 0.85,
)

print('HierarchicalCRLS (9 layers) instantiated.')
print(f'  Strategies: {STRATEGIES}')
print(f'  Decomp confidence base: 0.80 | Override threshold: 0.85')
print(f'  Layer hierarchy: WP17→WP19→WP20→WP21→WP22→WP23→WP24→WP25→WP26')

gen_log = []
accuracies, subtask_distributions, decomp_records = [], [], []

---
## Section 3 — Main Experiment: Observing Subtask Decomposition

In [ ]:
# ── 4. Main experiment loop ─────────────────────────────────────────────────
print('=' * 75)
print(f'  Gen  Regime        Acc     SubtaskTypes  ChosenAction      WP26Override')
print('=' * 75)

for gen in range(N_GENERATIONS):
    regime = REGIME_SEQUENCE[gen]
    puzzles = generate_puzzles(regime, PUZZLES_PER_GEN, BOARD_SIZE, seed=gen * 137 + 1)

    acc = stack.run_generation(puzzles)
    nine_tuple = stack.end_of_generation()
    (
        synth_rec, causal_rec, rollout_rec, meta_rec,
        explore_rec, distill_rec, ensemble_rec,
        forgetting_rec, decomp_rec
    ) = nine_tuple

    accuracies.append(acc)
    decomp_records.append(decomp_rec)

    # Extract subtask type distribution
    subtask_counts = Counter(s.subtask_type.value for s in decomp_rec.subtasks)
    subtask_distributions.append(subtask_counts)

    wp26_override = (decomp_rec.recombination_rule == 'decomposer_override')
    print(
        f'  {gen:3d}  {regime:<12}  {acc:.3f}  '
        f'{decomp_rec.n_subtask_types} types      '
        f'{decomp_rec.chosen_action.value:<18}  '
        f'{"OVERRIDE" if wp26_override else "---"}'
    )

print('=' * 75)
print(f'Mean accuracy: {np.mean(accuracies):.3f}')

In [ ]:
# ── 5. Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
gens = list(range(N_GENERATIONS))

regime_colors = {'ATARI': '#fff3cd', 'TERRITORY': '#d1ecf1', 'MIXED': '#e2d9f3'}

def add_regime_bands(ax):
    prev, start = None, 0
    for g, regime in enumerate(REGIME_SEQUENCE):
        if regime != prev:
            if prev is not None:
                ax.axvspan(start-0.5, g-0.5, alpha=0.25, color=regime_colors.get(prev, '#eee'))
            start, prev = g, regime
    ax.axvspan(start-0.5, N_GENERATIONS-0.5, alpha=0.25, color=regime_colors.get(prev, '#eee'))

# Panel A: Accuracy
ax = axes[0, 0]
add_regime_bands(ax)
ax.plot(gens, accuracies, 'g-o', linewidth=2.5, markersize=6)
ax.axhline(np.mean(accuracies), color='darkgreen', linestyle='--', alpha=0.7,
           label=f'Mean={np.mean(accuracies):.3f}')
ax.set_ylabel('Accuracy'); ax.set_title('Accuracy over Generations', fontweight='bold')
ax.legend(); ax.set_ylim(0, 1.05)

# Panel B: Number of distinct subtask types per generation
ax2 = axes[0, 1]
add_regime_bands(ax2)
n_types = [r.n_subtask_types for r in decomp_records]
ax2.bar(gens, n_types, color='#9C27B0', alpha=0.75, edgecolor='black')
ax2.set_ylabel('Distinct subtask types')
ax2.set_title('Subtask Type Diversity per Generation\n(MIXED regime → more types)', fontweight='bold')
ax2.set_yticks([0, 1, 2, 3, 4])

# Panel C: Subtask type distribution (stacked bar)
ax3 = axes[1, 0]
subtask_type_values = [st.value for st in SubtaskType]
subtask_colors = {'capture': '#FF5722', 'territory': '#2196F3', 'mixed': '#9C27B0', 'unknown': '#9E9E9E'}
bottoms = np.zeros(N_GENERATIONS)
for st_val in subtask_type_values:
    counts = np.array([d.get(st_val, 0) for d in subtask_distributions], dtype=float)
    if counts.sum() > 0:
        ax3.bar(gens, counts, bottom=bottoms, label=st_val.title(),
                color=subtask_colors.get(st_val, '#ccc'), alpha=0.8, edgecolor='black', linewidth=0.4)
        bottoms += counts
ax3.set_xlabel('Generation'); ax3.set_ylabel('Subtask record count')
ax3.set_title('Subtask Type Distribution per Generation\n(TaskDecomposer classification)', fontweight='bold')
ax3.legend(fontsize=9)

# Panel D: Chosen synthesis actions
ax4 = axes[1, 1]
chosen_actions = [r.chosen_action.value for r in decomp_records]
action_counts = Counter(chosen_actions)
colours = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']
bars = ax4.bar(list(action_counts.keys()), list(action_counts.values()),
               color=colours[:len(action_counts)], edgecolor='black')
ax4.bar_label(bars, fontsize=10)
ax4.set_xlabel('WP26 Chosen Action')
ax4.set_ylabel('Times chosen')
ax4.set_title('WP26 Synthesis Action Distribution\n(Decomposer recombination output)', fontweight='bold')
ax4.set_xticklabels([a.replace('_', '\n') for a in action_counts.keys()], fontsize=9)

fig.suptitle(
    'WP26: Hierarchical Task Decomposition — Prometheus v0\n'
    'Layer 9: Per-subtask synthesis recommendations via TaskDecomposer',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('wp26_hierarchical_decomp.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to wp26_hierarchical_decomp.png')

In [ ]:
# ── 6. Verify WP26 exit criteria ─────────────────────────────────────────────
results = verify_wp26_exit_criteria(stack)
print('WP26 Exit Criteria Verification')
print('=' * 50)
all_pass = True
for criterion, passed in results.items():
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f'  {status}  {criterion}')
    if not passed: all_pass = False
print()
if all_pass:
    print('All WP26 exit criteria satisfied.')
    print('The hierarchical decomposer is classifying puzzles and producing')
    print('per-subtask synthesis recommendations correctly.')
else:
    print('Some criteria not yet met — run more generations.')

---
## Conclusions

**Hierarchical task decomposition** (Sacerdoti 1977) allows the synthesis stack to
reason about *what kind* of problem each puzzle represents before committing to a
synthesis action. The `TaskDecomposer` provides a Hofstadterian *chunked abstraction*:
groups of structurally similar puzzles are handled as a unit, avoiding redundant
per-puzzle evaluation of the full 8-layer stack.

### References
- Sacerdoti, E.D. (1977). *A Structure for Plans and Behavior*. Elsevier.
- Hofstadter, D. (1979). *Gödel, Escher, Bach*. Basic Books.
- Newell, A. & Simon, H.A. (1972). *Human Problem Solving*. Prentice-Hall.
- Good, I.J. (1965). Speculations concerning the first ultraintelligent machine.